# Notebook 04: Master End-to-End Benchmark, Uncertainty Quantification & KPI Engine
**Complete All-in-One Pipeline**:
- Benchmarks all **7 Forecasting Models** (`Random Forest`, `Gradient Boosting`, `ARIMA / SARIMA`, `Exponential Smoothing`, `Naive Persistence`, `Moving Average 7d`, and `Mass-Balance Reconciled Ensemble`).
- Computes all **5 Executive Dashboard KPIs**:
  1. `Forecast Accuracy (%)`
  2. `Surge Lead Time (Days)`
  3. `Capacity Breach Probability (%)` (via 1,000-path Monte Carlo simulation)
  4. `Forecast Stability Index`
  5. `Model Robustness`
- Simulates **Multi-Scenario Policy Stress Tests** and generates the **30-Day Prescriptive Staffing & Shelter Bed Allocation Schedule**.

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from app import (
    load_and_prepare_data,
    engineer_features,
    run_walk_forward_backtest,
    generate_multi_horizon_forecasts,
)

df = load_and_prepare_data("../data/HHS_Unaccompanied_Alien_Children_Program.csv", "data/HHS_Unaccompanied_Alien_Children_Program.csv", "HHS_Unaccompanied_Alien_Children_Program.csv")
feat = engineer_features(df)

lb_care, pred_care, imp_care = run_walk_forward_backtest(feat, target="hhs_care", test_days=180)
lb_disc, pred_disc, imp_disc = run_walk_forward_backtest(feat, target="hhs_discharged", test_days=180)

print("==========================================================================================")
print("COMPLETE 7-MODEL LEADERBOARD: CHILDREN IN HHS CARE (180-DAY WALK-FORWARD VALIDATION)")
print("==========================================================================================")
print(lb_care[["Family", "MAE", "RMSE", "MAPE (%)", "Forecast Accuracy (%)", "Horizon 7d MAE", "Horizon 14d MAE", "Horizon 30d MAE", "Forecast Stability Index", "Model Robustness"]].to_string())

print("\n==========================================================================================")
print("COMPLETE 7-MODEL LEADERBOARD: DISCHARGE DEMAND (180-DAY WALK-FORWARD VALIDATION)")
print("==========================================================================================")
print(lb_disc[["Family", "MAE", "RMSE", "MAPE (%)", "Forecast Accuracy (%)", "Horizon 7d MAE", "Horizon 14d MAE", "Horizon 30d MAE", "Forecast Stability Index", "Model Robustness"]].to_string())

COMPLETE 7-MODEL LEADERBOARD: CHILDREN IN HHS CARE (180-DAY WALK-FORWARD VALIDATION)
                                             Family    MAE   RMSE  MAPE (%)  Forecast Accuracy (%)  Horizon 7d MAE  Horizon 14d MAE  Horizon 30d MAE  Forecast Stability Index  Model Robustness
Model                                                                                                                                                                                              
Random Forest Regressor            Machine Learning   4.61   5.94      0.21                  99.79            5.85             6.44             7.34                      98.7              99.8
ARIMA / SARIMA                          Statistical   4.82   6.21      0.22                  99.78            6.59             7.43             8.71                      95.5              99.8
Mass-Balance Reconciled Ensemble  Hybrid Reconciled   5.09   6.53      0.23                  99.77            6.46             7.11         

In [2]:
# Generate 30-Day Multi-Horizon Forecast & Executive KPI Summary
bundle_base = generate_multi_horizon_forecasts(
    feat=feat,
    horizon=30,
    selected_model="Mass-Balance Reconciled Ensemble",
    intake_multiplier=1.0,
    discharge_multiplier=1.0,
    capacity_threshold=2580.0,
)

bundle_surge = generate_multi_horizon_forecasts(
    feat=feat,
    horizon=30,
    selected_model="Mass-Balance Reconciled Ensemble",
    intake_multiplier=1.35,
    discharge_multiplier=1.0,
    capacity_threshold=2580.0,
)

print("=== EXECUTIVE DASHBOARD KPI SUMMARY (STATUS QUO vs +35% BORDER SURGE) ===")
kpi_table = pd.DataFrame({
    "Metric / KPI": [
        "Forecast Accuracy (%)",
        "Surge Lead Time (Days to 2,580 Cap)",
        "Capacity Breach Probability (%)",
        "Forecast Stability Index",
        "Model Robustness",
        "Expected 30-Day Peak Care Load",
    ],
    "Baseline (Status Quo)": [
        f"{lb_care.loc['Mass-Balance Reconciled Ensemble', 'Forecast Accuracy (%)']:.2f}%",
        str(bundle_base.surge_lead_time_days) + " Days" if bundle_base.surge_lead_time_days else ">30 Days (Safe)",
        f"{bundle_base.breach_probability_pct:.1f}%",
        f"{lb_care.loc['Mass-Balance Reconciled Ensemble', 'Forecast Stability Index']:.1f} / 100",
        f"{lb_care.loc['Mass-Balance Reconciled Ensemble', 'Model Robustness']:.1f} / 100",
        f"{bundle_base.expected_peak_care:,.1f} children",
    ],
    "Border Surge (+35% Intake)": [
        f"{lb_care.loc['Mass-Balance Reconciled Ensemble', 'Forecast Accuracy (%)']:.2f}%",
        str(bundle_surge.surge_lead_time_days) + " Days" if bundle_surge.surge_lead_time_days else ">30 Days (Safe)",
        f"{bundle_surge.breach_probability_pct:.1f}%",
        f"{lb_care.loc['Mass-Balance Reconciled Ensemble', 'Forecast Stability Index']:.1f} / 100",
        f"{lb_care.loc['Mass-Balance Reconciled Ensemble', 'Model Robustness']:.1f} / 100",
        f"{bundle_surge.expected_peak_care:,.1f} children",
    ]
})
print(kpi_table.to_string(index=False))
print("\nFirst 10 Days of Mass-Balance Reconciled Forecast Schedule:")
print(bundle_base.forecast_df.head(10).drop(columns=["Date"]).to_string())

=== EXECUTIVE DASHBOARD KPI SUMMARY (STATUS QUO vs +35% BORDER SURGE) ===
                       Metric / KPI Baseline (Status Quo) Border Surge (+35% Intake)
              Forecast Accuracy (%)                99.77%                     99.77%
Surge Lead Time (Days to 2,580 Cap)            21 Days                    12 Days
    Capacity Breach Probability (%)                 22.4%                      94.8%
           Forecast Stability Index            99.4 / 100                 99.4 / 100
                   Model Robustness            99.8 / 100                 99.8 / 100
     Expected 30-Day Peak Care Load      2,558.6 children           2,649.1 children

First 10 Days of Mass-Balance Reconciled Forecast Schedule:
            Forecast_HHS_Care  Care_CI80_Lower  Care_CI80_Upper  Care_CI95_Lower  Care_CI95_Upper  Forecast_Discharges  Discharge_CI80_Lower  Discharge_CI80_Upper  Discharge_CI95_Lower  Discharge_CI95_Upper  Reconciled_Transfers  Net_Intake_Exit_Imbalance  Implied_LoS_Days